In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
#import matplotlib
#import sys
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import umap
from matplotlib import pyplot as plt

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Conexión ACL

In [2]:


repo=Path(r"C:\Users\ACER\Desktop\UNIVERSITAS DOCTUS\Notas OneNote\Programación\Tesis\Encoder-xAI-LLMs\acl-anthology")

print(repo.exists())
xml_dir=repo/"data"/"xml"
print(xml_dir.exists())

xmls=list(xml_dir.glob("*.xml"))
print(len(xmls))
print(xmls[:5])

tree=ET.parse(xmls[0])
root=tree.getroot()
print(root.tag)

True
True
1619
[WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1952.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1956.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1957.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1960.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1961.earlymt.xml')]
collection


True
True
1619
[WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1952.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1956.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1957.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1960.earlymt.xml'), WindowsPath('C:/Users/ACER/Desktop/UNIVERSITAS DOCTUS/Notas OneNote/Programación/Tesis/Encoder-xAI-LLMs/acl-anthology/data/xml/1961.earlymt.xml')]
collection

1619 es el número de colecciones, no de artículos. El hecho que root.tag imprima collection lo confirma.


# Mirar dentro colección

In [13]:
#tree=ET.parse(xmls[0])
#root=tree.getroot()
print(root.attrib)
for child in root[:5]:
    print(child.tag)

primer_hijo=root[0]
print(primer_hijo.tag)
print(primer_hijo.attrib)

{'id': '1952.earlymt'}
volume
volume
{'id': '1', 'ingest-date': '2021-10-24', 'type': 'proceedings'}


In [3]:
vol=root[0]
print(vol.tag)
print(vol.attrib)
'''for child in vol[:10]:
    print(child.tag)'''
paper=vol.find("paper")
print(paper.attrib)
for child in paper:
    print(child.tag)

volume
{'id': '1', 'ingest-date': '2021-10-24', 'type': 'proceedings'}
{'id': '1'}
title
author
url
bibkey


# Inspección de paper


In [4]:
paper=vol.find("paper")
print(paper.attrib)
for child in paper:
    print(f'{child.tag}||{child.text}')
    print('-'*50)
# Parece que los papers antiguos no vienen con abstract (solo: title,author,url,bibkey)

{'id': '1'}
title||Translation
--------------------------------------------------
author||None
--------------------------------------------------
url||1952.earlymt-1.1
--------------------------------------------------
bibkey||weaver-1952-translation
--------------------------------------------------


In [27]:
newtree=ET.parse(xmls[-1])
newroot=newtree.getroot()
newvol=newroot.find("volume")
newpaper=newvol.find("paper")
for child in newpaper:
    print(child.tag)
print('-'*10)    
for child in newvol[:3]:
    print("\nPAPER")
    print(child.tag)

    if child.tag=="paper":
        for x in child:
            print("   ",x.tag)

title
author
pages
url
doi
bibkey
----------

PAPER
meta

PAPER
frontmatter

PAPER
paper
    title
    author
    pages
    url
    doi
    bibkey


# Extractor

In [5]:
papers=[]

for xml_file in xmls:

    tree=ET.parse(xml_file)
    root=tree.getroot()

    for vol in root.findall("volume"):

        year=None

        meta=vol.find("meta")

        if meta is not None:

            year_tag=meta.find("year")

            if year_tag is not None:
                year=year_tag.text

        for paper in vol.findall("paper"):

            title=paper.findtext("title")
            abstract=paper.findtext("abstract")
            doi=paper.findtext("doi")
            url=paper.findtext("url")
            bibkey=paper.findtext("bibkey")

            papers.append(
                {
                    "title":title,
                    "abstract":abstract,
                    "year":year,
                    "doi":doi,
                    "url":url,
                    "bibkey":bibkey
                }
            )


In [6]:
df=pd.DataFrame(papers)
#print(df.info)
print(df.columns)
print(df["abstract"].notna().mean())

Index(['title', 'abstract', 'year', 'doi', 'url', 'bibkey'], dtype='object')
0.6229353944383769


In [7]:
df["year"]=pd.to_numeric(df["year"],errors="coerce")
df.groupby(
    pd.cut(
        df["year"],
        bins=[1950,1980,1990,2000,2010,2020,2030]
        )
        )["abstract"].apply(
            lambda x:x.notna().mean()
            )

C:\Users\ACER\AppData\Local\Temp\ipykernel_4560\3967937327.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


year
(1950, 1980]    0.073457
(1980, 1990]    0.023599
(1990, 2000]    0.063967
(2000, 2010]    0.194091
(2010, 2020]    0.544505
(2020, 2030]    0.933147
Name: abstract, dtype: float64

## Subconjunto

In [8]:
df_recent = df[
    (df["year"] >= 2020)
    &
    (df["abstract"].notna())
].copy()

df_xai = df_recent[
    (
        df_recent["title"].fillna("")
        + " "
        + df_recent["abstract"].fillna("")
    ).str.contains(
        r"explain|interpret|probing|causal|counterfactual|attribution|mechanistic",
        case=False,
        regex=True
    )
]
len(df_xai)

6623

In [9]:
#print(df_xai.columns)
df_xai["text"]=df["title"]+"."+df_xai["abstract"]
#print(df_xai.columns)
print(df_xai.index)
print(df_xai.loc[6989:6996,["text"]])

Index([ 6989,  6996,  6997,  7015,  7017,  7018,  7030,  7043,  7060,  7064,
       ...
       66606, 66611, 66624, 66626, 66628, 66650, 66656, 66662, 66664, 66665],
      dtype='int64', length=6623)
                                                   text
6989  .We propose a novel, accurate, and explainable...
6996  Towards a Better Understanding of Label Smooth...


C:\Users\ACER\AppData\Local\Temp\ipykernel_4560\1866090167.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_xai["text"]=df["title"]+"."+df_xai["abstract"]


# Embeddings

In [10]:
model=SentenceTransformer("all-MiniLM-L6-v2")
embeddings=model.encode( # numpy array
    df_xai["text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 207/207 [05:12<00:00,  1.51s/it]


In [17]:
print(embeddings.shape)
print(df_xai.shape)
print(f"size: {df_xai.size}")

(6623, 384)
(6623, 7)
size: 46361


In [12]:

query1 = model.encode(
    "causal explanation methods"
)

score1 = cosine_similarity( #Crea un numpy array con el porcentaje de similitud de cada articulo.
    [query1],
    embeddings
)[0]
top1 = np.argsort(score1)[::-1] # Ordeno los indices del score más alto al más bajo.

In [39]:
#print(f'shape top: {top1.shape}') # Top1  es un array (1D/6623) con los índices ordenados de los artículos del más similar al menos.
#print(f'shape score: {score1.shape}\nshape embedd: {embeddings.shape} ') #score1 es un array (1D/6623) con porcentaje de similitud (embedding-query)
print(f'top1: {top1}') # Top1  es un array (1D/6623) con los índices ordenados de los artículos del más similar al menos.
print(f'score1: {score1}') #score1 es un array (1D/6623) con porcentaje de similitud (embedding-query) desordenado
print(f'score promedio: {score1.mean()}')
print(f'score máximo: {score1.max()}')
print(f'\nScore del 1 top: {score1[top1[0]]}') 
print(df_xai.iloc[top1[0]])

top1: [ 640  511 3784 ... 3365  431 4721]
score1: [0.20340332 0.07082733 0.21138233 ... 0.08385    0.21174125 0.05232011]
score promedio: 0.16511735320091248
score máximo: 0.6563277244567871

Score del 1 top: 0.6563277244567871
title                                                        
abstract    Prior work infers the causation between events...
year                                                     2021
doi                             10.18653/v1/2021.acl-long.183
url                                         2021.acl-long.183
bibkey                                     du-etal-2021-excar
text        .Prior work infers the causation between event...
Name: 14242, dtype: object


In [40]:

query2 = model.encode(
    "probing methods"
)

score2 = cosine_similarity( # 
    [query2],
    embeddings
)[0]
top2 = np.argsort(score2)[::-1] # 

In [ ]:
print(f'score promedio: {score2.mean()}') #score2 tiene la misma estructura que score1
print(f'score máximo: {score2.max()}')
print(f'\nScore del 1 top: {score2[top2[0]]}') #top2 tiene la misma estructura que top2
print(df_xai.iloc[top2[0]])

score promedio: 0.084694042801857
score máximo: 0.508607029914856

Score del 1 top: 0.508607029914856
title                         Does My Representation Capture 
abstract    Probing (or diagnostic classification) has bec...
year                                                     2021
doi                              10.18653/v1/2021.acl-demo.23
url                                          2021.acl-demo.23
bibkey                      ferreira-etal-2021-representation
text        Does My Representation Capture .Probing (or di...
Name: 14828, dtype: object


In [43]:

query3 = model.encode(
    "feature attribution methods"
)

score3 = cosine_similarity(
    [query3],
    embeddings
)[0]

top3 = np.argsort(score3)[::-1]

In [44]:
print(f'score promedio: {score3.mean()}') #score2 tiene la misma estructura que score1
print(f'score máximo: {score3.max()}')
print(f'\nScore del 1 top: {score3[top3[0]]}') #top2 tiene la misma estructura que top2
print(df_xai.iloc[top3[0]])

score promedio: 0.21815577149391174
score máximo: 0.5706121921539307

Score del 1 top: 0.5706121921539307
title            Source Attribution for Large Language Models
abstract    As Large Language Models (LLMs) become more wi...
year                                                     2025
doi                       10.18653/v1/2025.ijcnlp-tutorials.1
url                                   2025.ijcnlp-tutorials.1
bibkey                                 rawte-etal-2025-source
text        Source Attribution for Large Language Models.A...
Name: 61030, dtype: object
